In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# ==========================================
# 1. Automatic Environment Detection
# ==========================================
try:
    # --- Colab Environment ---
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY') or ""
    print("☁️ Colab Environment Detected.")
except ImportError:
    # --- Local Environment (VS Code) ---
    from dotenv import load_dotenv
    current_dir = os.getcwd()
    project_root = os.path.abspath(os.path.join(current_dir, "..", ".."))
    env_path = os.path.join(project_root, ".env")
    load_dotenv(dotenv_path=env_path, override=True)
    print("💻 Local Environment Detected.")

# ==========================================
# 2. Define Components & Build the Chain
# ==========================================
# Initialize the model (Equivalent to ChatOpenAI in the lesson)
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.1
)

# Define prompt and custom lambda function
prompt = ChatPromptTemplate.from_template("شهر {city} در کدام کشور است؟ فقط نام کشور را بگو.")
to_upper = RunnableLambda(lambda x: x.upper())

# Build the LCEL Chain
chain = prompt | model | StrOutputParser() | to_upper
print("✅ Chain is successfully built!")

💻 Local Environment Detected.
✅ Chain is successfully built!


In [3]:
from langchain_core.globals import set_debug

# ==========================================
# Level 1: Console Debugging
# ==========================================

# 1. Turn on debug mode
set_debug(True)

print("⏳ Executing chain with DEBUG MODE ON...\n")
response = chain.invoke({"city": "پاریس"})
print(f"\n🎯 Final Output: {response}")

# 2. Turn off debug mode to prevent cluttering future executions
set_debug(False)

⏳ Executing chain with DEBUG MODE ON...

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "city": "پاریس"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "city": "پاریس"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[llm/start] [chain:RunnableSequence > llm:ChatGoogleGenerativeAI] Entering LLM run with input:
{
  "prompts": [
    "Human: شهر پاریس در کدام کشور است؟ فقط نام کشور را بگو."
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatGoogleGenerativeAI] [1.32s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "فرانسه",
        "generation_info": {
          "finish_reason": "STOP",
          "model_name": "gemini-3.5-flash-lite",
          "safety_ratings": []
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content": [
              {
                "type": "text",
                "text": "فرانسه",
                "extras": {
                  "signature": "EmAKXgFpFH0ThugsnBVm8N7okTCEJSCYdf5WrRcx/+nb576rI351sVIh7w4yLvQxGeM9RlZakteDVm3cJ

In [4]:
# Install grandalf quietly if not already installed (Required for ASCII graph)
!pip install -q grandalf

# ==========================================
# Level 2: Visualizing the Chain Architecture
# ==========================================

print("--- 🗺️ Chain Architecture Graph ---\n")

# Print the ASCII representation of the LCEL pipeline
chain.get_graph().print_ascii()

--- 🗺️ Chain Architecture Graph ---

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
        +--------+         
        | Lambda |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | LambdaOutput |      
     +--------------+      


In [5]:
pip install -U langchain langchain-google-genai langsmith

Note: you may need to restart the kernel to use updated packages.


In [6]:
# ==========================================
# Level 3: Professional Tracing with LangSmith
# ==========================================

# 1. Enable Tracing globally
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# 2. Check if the LangSmith API Key is provided
if not os.environ.get("LANGCHAIN_API_KEY"):
    print("⚠️ Warning: LANGCHAIN_API_KEY is not set.")
    print("To see tracing in action, create an account at smith.langchain.com,")
    print("generate an API key, and add it to your .env file as LANGCHAIN_API_KEY='ls_...'\n")
else:
    print("✅ LangSmith Tracing is ENABLED.")

# 3. (Optional) Set the project name for the LangSmith dashboard
os.environ["LANGCHAIN_PROJECT"] = "My First LangChain Project"

print("⏳ Executing chain to send trace logs to LangSmith...\n")

# Execute the chain normally. LangChain automatically sends logs to the background.
response = chain.invoke({"city": "توکیو"})
print(f"🎯 Final Output: {response}")

print("\n🌐 If your LangSmith API key is valid, check your dashboard at smith.langchain.com!")

✅ LangSmith Tracing is ENABLED.
⏳ Executing chain to send trace logs to LangSmith...

🎯 Final Output: ژاپن

🌐 If your LangSmith API key is valid, check your dashboard at smith.langchain.com!
